In [48]:
import os
from typing import List
from pydantic import BaseModel, Field
from langchain_classic.schema import Document
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.tools import WikipediaQueryRun,ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper
from langgraph.graph import StateGraph,END,START
from langgraph.graph.message import AnyMessage, add_messages
from typing_extensions import Annotated
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.docstore.in_memory import InMemoryDocstore

from langgraph.checkpoint.memory import MemorySaver

import requests
from langchain.tools import tool

from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_community.vectorstores import FAISS

from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline, EmbeddingsFilter

from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.tools import TavilySearchResults
from IPython.display import Image, display

from langchain_community.tools.pubmed.tool import PubmedQueryRun
from langchain_exa import ExaSearchRetriever

import faiss

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY")

In [46]:
VECTOR_DIM = 384

In [2]:
local_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
llm=ChatGroq(model="qwen/qwen3-32b",temperature=0)

In [3]:
urls = [
    "https://python.langchain.com/docs/introduction/",
    "https://python.langchain.com/docs/tutorials/llm_chain/",
    "https://python.langchain.com/docs/concepts/",
    "https://python.langchain.com/docs/tutorials/rag/"
]

In [4]:
import unicodedata
import re
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
raw_docs=[WebBaseLoader(url).load() for url in urls]

In [6]:
raw_docs

[[Document(metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain is an open source framework with a pre-built agent architecture and integrations for any model or tool — so you can build agents that adapt as fast as the ecosystem evolves', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageLangChain + LangGraphSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewLangChainLangGraphDeep AgentsIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewBuilt-in middlewareCustom middlewareAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat

In [8]:
def clean_text(text: str) -> str:
    # Normalize unicode characters (NFKD decomposes combined characters)
    print("---cleaning text---⚽")
    text = unicodedata.normalize('NFKD', text)
    
    # Replace common ligatures
    ligatures = {
        '\uFB00': 'ff', '\uFB01': 'fi', '\uFB02': 'fl',
        '\uFB03': 'ffi', '\uFB04': 'ffl', '\uFB05': 'st', '\uFB06': 'st'
    }
    for char, replacement in ligatures.items():
        text = text.replace(char, replacement)
    
    # Remove non-printable characters
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] != "C")

    # Remove extra whitespace and newlines
    cleaned_text = ' '.join(text.split())
    
    return cleaned_text

In [9]:
def splitter(text: str, threshold: float = 0.7):
    # Consider using a more robust sentence splitter than just \n
    print("---splitting text---⚽")
    sentences = [s.strip() for s in text.split(". ") if s.strip()]
    if not sentences:
        return []

    embeddings = local_embeddings.embed_documents(sentences)
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        # Calculate similarity between current sentence and the previous one
        # Note: Some prefer comparing current sentence to the average of the current_chunk
        sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]

        if sim > threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(". ".join(current_chunk))
            current_chunk = [sentences[i]]
    
    # Append the final remaining chunk
    if current_chunk:
        chunks.append(". ".join(current_chunk))
        
    return chunks

In [10]:
def LoadDocuments(urls: List[str]) -> List[Document]:
    # 1. Load all URLs - this returns a list of lists: [[Doc1], [Doc2]]
    
    raw_site_data = [WebBaseLoader(url).load() for url in urls]

    ProcessedDocs = []
    
    # 2. Iterate through each site's loaded content
    # site_docs is the list of Documents returned for ONE url
    for site_docs in raw_site_data:
        # Get the first Document object from the list
        current_doc = site_docs[0]
        
        # Extract base metadata (like source and title) from the loader
        base_meta = current_doc.metadata 
        
        # Clean the content using your clean_text function
        cleaned_text = clean_text(current_doc.page_content)

        # 3. Merge base metadata with your custom tags
        meta_data = {
            **base_meta,    
            "language": "python",
            "framework": "LangGraph",
            "category": "Documentation/Tutorial",
            "topics": [
                "State Management", "Nodes and Edges", 
                "Multi-agent Orchestration", "Tool Calling"
            ],
            "version": "1.0",
            "author": "LangChain Inc."
        }
        
        ProcessedDocs.append(Document(page_content=cleaned_text, metadata=meta_data))

    ## 4. Semantic Chunking
    result = []
    for doc in ProcessedDocs:
        # This calls your splitter which uses cosine similarity
        chunks = splitter(doc.page_content)
        for chunk in chunks:
            # We attach the parent document's metadata to every chunk
            result.append(Document(page_content=chunk, metadata=doc.metadata))
            
    return result

In [11]:
results=LoadDocuments(urls)

---cleaning text---⚽
---cleaning text---⚽
---cleaning text---⚽
---cleaning text---⚽
---splitting text---⚽
---splitting text---⚽
---splitting text---⚽
---splitting text---⚽


In [12]:
results

[Document(metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain is an open source framework with a pre-built agent architecture and integrations for any model or tool — so you can build agents that adapt as fast as the ecosystem evolves', 'language': 'python', 'framework': 'LangGraph', 'category': 'Documentation/Tutorial', 'topics': ['State Management', 'Nodes and Edges', 'Multi-agent Orchestration', 'Tool Calling'], 'version': '1.0', 'author': 'LangChain Inc.'}, page_content='LangChain overview - Docs by LangChainSkip to main contentDocs by LangChain home pageLangChain + LangGraphSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewLangChainLangGraphDeep AgentsIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewBuilt

HYBRID RETRIEVER FOR BETTER RETRIEVAL

In [13]:
##making dense retriever
vectorstore = FAISS.from_documents(results, local_embeddings)

In [14]:
dense_retriever=vectorstore.as_retriever(search_type="mmr", search_kwargs={"k":3,"fetch_k":7,})

sparce_retriever=BM25Retriever.from_documents(results)
sparce_retriever.k=3

In [15]:
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever, sparce_retriever],
    weights=[0.7, 0.3]
)

In [16]:
result=hybrid_retriever.invoke("what is langchain")

###EXPERIMENTAL GLOBAL MMR FILTERING

In [17]:
redundant_filter = EmbeddingsRedundantFilter(embeddings=local_embeddings)
relevant_filter = EmbeddingsFilter(embeddings=local_embeddings, similarity_threshold=0.75)

pipeline_compressor = DocumentCompressorPipeline(
    transformers=[redundant_filter,relevant_filter]
)

# 3. Now pass the pipeline to the retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, 
    base_retriever=hybrid_retriever
)

In [18]:
results=compression_retriever.invoke("what is langchain")

SETTING UP TAVILLY TOOL AND OTHER TOOLS FOR THE NODES TO WORK WITH

In [19]:
from langchain_community.tools.tavily_search import TavilySearchResults

medical_genomic_tool = TavilySearchResults(
    max_results=5,
    search_depth="advanced", # Crucial for scientific precision
    include_answer=True,    # Tavily's AI will pre-summarize findings
    include_raw_content=True, # Gives your LLM the full cleaned text to analyze
    include_domains=[
        "ncbi.nlm.nih.gov",  # PubMed / PMC / ClinVar
        "nature.com",        # Scientific Journals
        "genome.gov",        # National Human Genome Research Institute
        "ensembl.org",       # Genome Browser
        "omim.org",          # Online Mendelian Inheritance in Man
        "cell.com",          # Life sciences research
        "bioinformatics.org"
    ]
)
medical_genomic_tool.invoke("tell me something about BRCA1 gene")

C:\Users\ppriy\AppData\Local\Temp\ipykernel_20712\1323943078.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  medical_genomic_tool = TavilySearchResults(


[{'title': 'BRCA Genes: The Role in Genome Stability, Cancer ...',
  'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6548160/',
  'content': 'Schematic representation of functional domains within BRCA1 and BRCA2 proteins and the position of several founder mutations. BRCA1 is composed of 23 exons and BRCA2 includes 27 exons. Both genes encode large proteins: BRCA1 consists of 1,863 amino acids and BRCA2 of 3,418 amino acids. BRCA1 has a highly conserved zinc-binding RING (really interesting new gene) finger domain which is located close to the N-terminus. At the C-terminus, two BRCT (BRCA1 C-terminal) domains are located. The central part of BRCA1 consists of two NLS (nuclear localization signals) and one coiled coil domain. BRCA2 contains eight copies of a 20-30 amino acid repeat, termed BRC repeats. At the amino-terminus, BRCA2 has a TAD (transcriptional activation domain) domain and at the carboxyl-terminus two NLS and one [...] More than 1600 mutations were identified in BRCA1 gen

ADDITIONAL TOOLS FOR PUBMED AND EXA SEARCH RETRIEVER

setup 1: Building the Case (pubmed_tool)

Now that you have a dangerous variant, the system looks for established medical consensus.

    Action: It uses the PubmedQueryRun wrapper to fetch the most relevant abstracts.

    Result: It provides up to 5 peer-reviewed summaries that explain the clinical phenotype and history of those mutations.

    Why: This gives the agent scientific "authority" backed by the National Library of Medicine.


Setup 2: The 2026 Edge (exa_retriever)

Finally, the system checks for what hasn't been "officially" indexed yet.

    Action: It performs a neural search across high-authority domains like nature.com and cell.com.

    Result: It pulls 2025-2026 research snippets and trial updates (like the neoadjuvant trials you found earlier).

    Why: It ensures the agent knows about breakthroughs that happened just weeks or months ago.    

In [20]:
pubmed_tool= PubmedQueryRun(
    max_results=5 
)

exa_retriever = ExaSearchRetriever(
    k=3, 
    highlights=True, 
    exa_api_key=os.getenv("EXA_API_KEY"),
    include_domains=["nature.com", "cell.com", "genome.gov"]
)


Phase 1: Physical Grounding (Ensembl)

The journey starts by identifying where the gene lives in the human body.

    The Action: The agent sends the gene name (e.g., BRCA1 or CASQ2) to the Ensembl Tool.

    The Output: It retrieves the specific Chromosome, the Start/End coordinates, and the Strand orientation.

    The "Why": This ensures the agent is looking at the correct part of the genome and isn't just "guessing" based on names.

In [21]:
import requests
from langchain.tools import tool

@tool
def ensembl_gene_lookup(gene_symbol: str) -> str:
    """
    Fetches the chromosome, start/end coordinates, and Ensembl ID for a human gene.
    Use this to verify exactly where a gene sits on the genome.
    """
    server = "https://rest.ensembl.org"
    ext = f"/lookup/symbol/homo_sapiens/{gene_symbol}?"
    
    try:
        r = requests.get(server + ext, headers={"Content-Type": "application/json"})
        if not r.ok:
            return f"Ensembl could not find data for {gene_symbol}."
        
        data = r.json()
        return (
            f"Gene: {gene_symbol}\n"
            f"Ensembl ID: {data['id']}\n"
            f"Location: Chromosome {data['seq_region_name']} "
            f"({data['start']}-{data['end']})\n"
            f"Strand: {'Forward' if data['strand'] == 1 else 'Reverse'}"
        )
    except Exception as e:
        return f"Ensembl Request Failed: {str(e)}"

In [22]:
pubmed_tool.invoke("latest research on CRISPR gene editing")

'Published: 2026-01-19\nTitle: Prokaryotic Molecular Defense Mechanisms and Their Potential Applications in Cancer Biology: A Special Consideration for Cyanobacterial Systems.\nCopyright Information: \nSummary::\nCyanobacteria harbor sophisticated molecular defense systems that have evolved over billions of years to protect against viral invasion and foreign genetic elements. These ancient photosynthetic organisms possess a diverse array of restriction-modification (R-M) systems and CRISPR-Cas arrays that present challenges for genetic engineering, but also offer unique opportunities for cancer-targeted biotechnological applications. These systems exist in prokaryotes mainly as defense mechanisms but they are currently used in molecular applications as gene editing tools. Moreover, latest developments in nucleases such as zinc finger nucleases (ZFNs), TALENs (transcription-activator-like effector nucleases) are discussed. A comprehensive genomic analysis of 126 cyanobacterial species f

In [23]:
exa_retriever.invoke("tell me about loss of function during genetic analysis and how it can detect breast cancer")

[Document(metadata={'title': 'Functional evaluation and clinical classification of BRCA2 variants', 'url': 'https://www.nature.com/articles/s41586-024-08388-8?error=cookies_not_supported&code=a15516de-fe1a-40c3-9a5b-5a00398df8f5', 'id': 'https://www.nature.com/articles/s41586-024-08388-8?error=cookies_not_supported&code=a15516de-fe1a-40c3-9a5b-5a00398df8f5', 'score': 0.38319718837738037, 'published_date': '2025-01-08T00:00:00.000Z', 'author': 'Couch, Fergus J.', 'highlights': ['_BRCA2_ is an established clinically actionable cancer predisposition gene[5] and has been widely used to test for hereditary cancer risk. In particular, _BRCA2_ loss-of-function pathogenic variants are associated with a 69% lifetime risk of developing breast cancer[2] and a 15% risk of developing ovarian cancer[4]. The risk of developing pancreatic cancer', 'Germline _BRCA2_ loss-of function variants, which can be identified through clinical genetic testing, predispose to several cancers[1], [2], [3], [4], [5].

Step 2: Finding the "Suspects" (clinvar_lookup)

This is your new discovery tool. It finds which specific variants of that gene have been reported.

    Action: It searches the ClinVar database for any variants linked to that gene symbol.

    Result: It retrieves a list of Top IDs (Unique Identifiers) for those variants.

    Why: You can't get medical details without first having the specific IDs to look up.

In [24]:
@tool
def clinvar_lookup(gene_symbol: str) -> str:
    """Useful for finding clinical significance and pathogenicity of variants for a gene."""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "clinvar",
        "term": f"{gene_symbol}[gene]",
        "retmode": "json",
        "retmax": 5
    }
    # For higher volume, add: "api_key": os.getenv("NCBI_API_KEY")
    resp = requests.get(base_url, params=params)
    data = resp.json()
    ids = data.get("esearchresult", {}).get("idlist", [])
    
    if not ids:
        return f"No ClinVar entries found for {gene_symbol}."
    
    return f"Found {len(ids)} recent ClinVar records for {gene_symbol}. Top IDs: {', '.join(ids)}"

In [25]:
print("Testing ClinVar...")
clinvar_result = clinvar_lookup.invoke("BRCA1")
print(f"ClinVar Output: {clinvar_result}\n")

Testing ClinVar...
ClinVar Output: Found 5 recent ClinVar records for BRCA1. Top IDs: 4687778, 4686574, 4686566, 4685876, 4684220



Step 3: The High-Priority Filter (clinvar_details)

This is the "Brain" of your clinical logic. It sifts through the IDs found in Step 2.

    Action: It takes those IDs and looks deep into their nested JSON data.

    The Filter: It ignores the "noise" (Benign/Uncertain results) and only triggers a 🚨 HIGH PRIORITY alert if the status is Pathogenic or Likely Pathogenic.

    Result: A focused report of the actual "Red Flags."

In [26]:
import requests
from langchain.tools import tool

@tool
def clinvar_details(id_list: str) -> str:
    """
    Fetches ClinVar details and FILTERS for Pathogenic/Likely Pathogenic variants.
    Useful for high-priority clinical reports.
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
    params = {
        "db": "clinvar",
        "id": id_list,
        "retmode": "json"
    }
    
    resp = requests.get(base_url, params=params)
    data = resp.json()
    uids = data.get("result", {}).get("uids", [])
    
    reports = []
    # Target values for our high-priority filter
    target_pathogenicity = ["pathogenic", "likely pathogenic"]

    for uid in uids:
        item = data["result"][uid]
        title = item.get("title", f"Variation {uid}")
        
        # 1. Check Germline (Inherited)
        germline = item.get("germline_classification", {})
        pathogenicity = germline.get("description", "Not Reported")
        condition = germline.get("trait_name", "Condition Not Listed")

        # 2. Check Somatic (Oncogenicity/Clinical Impact) fallback
        if pathogenicity == "Not Reported":
            somatic = item.get("oncogenicity_classification", {})
            pathogenicity = somatic.get("description", "Not Reported")
            condition = somatic.get("trait_name", condition)

        # 3. APPLY THE FILTER
        if pathogenicity.lower() in target_pathogenicity:
            reports.append(
                f"🚨 HIGH PRIORITY: {title}\n"
                f"   - PATHOGENICITY: {pathogenicity}\n"
                f"   - CONDITION: {condition}\n"
                f"   - UID: {uid}"
            )

    if not reports:
        return "No Pathogenic or Likely Pathogenic variants found in this batch."

    return "\n\n".join(reports)

In [27]:
# This should show the 🚨 icon because it is Pathogenic
print(clinvar_details.invoke("10")) 

# This should return "No Pathogenic variants found" for your previous VUS ID
print(clinvar_details.invoke("17610"))

No Pathogenic or Likely Pathogenic variants found in this batch.
🚨 HIGH PRIORITY: NM_001232.4(CASQ2):c.919G>C (p.Asp307His)
   - PATHOGENICITY: Pathogenic
   - CONDITION: Condition Not Listed
   - UID: 17610


PIPELINE TESTING

In [28]:
# Investigating the BAP1 conflict from your images
def investigate_bap1_conflict(gene="BAP1", variant_id="4646508"):
    print(f"🔬 INVESTIGATING CLINICAL CONFLICT: {gene}")
    print("="*60)

    # 1. Physical Mapping (from your Image 2)
    # Target: Chromosome 12 (111,642,145-111,685,955)
    print("\n[1] Mapping BAP1 coordinates...")
    location = ensembl_gene_lookup.invoke(gene)
    print(f"✅ {location}")

    # 2. The VUS Check (from your Image 3)
    print(f"\n[2] Checking ClinVar VUS status for ID {variant_id}...")
    # This will return your 'No Pathogenic variant' alert, 
    # which confirms the conflict seen in your screenshot.
    analysis = clinvar_details.invoke(variant_id)
    print(f"🩺 ClinVar Status: {analysis}")

    # 3. Resolving the Conflict with 2026 Data
    print("\n[3] Searching for Evo2-related functional studies (2025-2026)...")
    # We use Exa to find why the computational tool (Evo2) suggests it is functional.
    research_query = f"BAP1 variant 111650033 A>T functional impact research 2026"
    breakthroughs = exa_retriever.invoke(research_query)

    print("\n" + "="*60)
    if breakthroughs:
        print(f"🌐 NEW EVIDENCE FOUND:\n{breakthroughs[0].page_content[:400]}...")
    else:
        print("❌ No specific reclassification papers found yet in the 2026 index.")

investigate_bap1_conflict()

🔬 INVESTIGATING CLINICAL CONFLICT: BAP1

[1] Mapping BAP1 coordinates...
✅ Gene: BAP1
Ensembl ID: ENSG00000163930
Location: Chromosome 3 (52401008-52410008)
Strand: Reverse

[2] Checking ClinVar VUS status for ID 4646508...
🩺 ClinVar Status: No Pathogenic or Likely Pathogenic variants found in this batch.

[3] Searching for Evo2-related functional studies (2025-2026)...

🌐 NEW EVIDENCE FOUND:
Saturation genome editing of BAP1 functionally classifies somatic and germline variants | Nature Genetics
[Skip to main content] 
Thank you for visiting nature.com. You are using a browser version with limited support for CSS. To obtain
the best experience, we recommend you use a more up to date browser (or turn off compatibility mode in
Internet Explorer). In the meantime, to ensure continued sup...


In [30]:
import os
from dotenv import load_dotenv

# 1. Setup
load_dotenv()

def breast_cancer_investigator(gene="BRCA1", variant_id="17610"):
    print(f"🧬 BREAST CANCER PIPELINE TEST: {gene}")
    print("="*60)

    # --- STEP 1: PHYSICAL GROUNDING (Aligns with Image 2) ---
    # We verify the location on Chromosome 17
    print(f"\n[1] Verifying {gene} Coordinates (Chr 17)...")
    location = ensembl_gene_lookup.invoke(gene)
    print(f"📍 {location}")

    # --- STEP 2: CLINICAL STATUS (Aligns with Image 3) ---
    # We check the status of the specific breast cancer variant
    print(f"\n[2] Analyzing Pathogenicity (ID {variant_id})...")
    analysis = clinvar_details.invoke(variant_id)
    print(f"🩺 {analysis}")

    # --- STEP 3: ENHANCED RESEARCH (The 2026 Edge) ---
    # We use Exa and PubMed to find the latest targeted therapy data
    if "🚨 HIGH PRIORITY" in analysis:
        print(f"\n[3] Querying 2026 Research for Targeted Therapies...")
        
        # Searching for Nature/Cell breakthroughs from 2026
        research_query = f"latest 2026 PARP inhibitor trials for {gene} positive breast cancer"
        breakthroughs = exa_retriever.invoke(research_query)

        # --- FINAL FRONTEND-ALIGNED DASHBOARD ---
        print("\n" + "="*60)
        print(f"## 🧬 GENOMIC ANALYSIS REPORT: {gene}")
        print(f"> **Baseline Status:** {analysis.splitlines()[0]}")
        print(f"**Physical Address:** {location.splitlines()[2]}")
        
        print("\n### 🌐 2026 Evidence Synthesis")
        if breakthroughs:
             # Mirroring the nature.com snippets from your previous outputs
             print(f"**Key Breakthrough:** {breakthroughs[0].page_content[:450]}...")
        else:
             print("No 2026 breakthroughs found for this specific query.")
    
    print("\n---")
    print("**Disclaimer:** For research and educational use only.")

# Run the breast cancer test
breast_cancer_investigator()

🧬 BREAST CANCER PIPELINE TEST: BRCA1

[1] Verifying BRCA1 Coordinates (Chr 17)...
📍 Gene: BRCA1
Ensembl ID: ENSG00000012048
Location: Chromosome 17 (43044292-43170245)
Strand: Reverse

[2] Analyzing Pathogenicity (ID 17610)...
🩺 🚨 HIGH PRIORITY: NM_001232.4(CASQ2):c.919G>C (p.Asp307His)
   - PATHOGENICITY: Pathogenic
   - CONDITION: Condition Not Listed
   - UID: 17610

[3] Querying 2026 Research for Targeted Therapies...

## 🧬 GENOMIC ANALYSIS REPORT: BRCA1
> **Baseline Status:** 🚨 HIGH PRIORITY: NM_001232.4(CASQ2):c.919G>C (p.Asp307His)
**Physical Address:** Location: Chromosome 17 (43044292-43170245)

### 🌐 2026 Evidence Synthesis
**Key Breakthrough:** Neoadjuvant PARP inhibitor scheduling in BRCA1 and BRCA2 related breast cancer: PARTNER, a randomized phase II/III trial | Nature Communications
[Skip to main content] 
Thank you for visiting nature.com. You are using a browser version with limited support for CSS. To obtain
the best experience, we recommend you use a more up to date 

making the schema 


In [29]:
from typing import TypedDict, List, Optional, Annotated
import operator

class GenomicArbiterState(TypedDict):
    # Inputs from your frontend (Image 2 & 3)
    gene_symbol: str        # e.g., "BRAP"
    variant_id: str         # e.g., "4646508"
    evo2_prediction: str    # e.g., "FUNC"
    
    # State updates from each node execution
    physical_map: Optional[str]
    discovered_ids: Optional[str]
    clinvar_report: Optional[str]
    
    # Accumulated research (operator.add lets parallel tools append to this list)
    research_findings: Annotated[List[str], operator.add]
    
    # Final output for the UI
    final_audit_report: Optional[str]
    clinical_recommendations: Optional[str]  # ADD THIS
    confidence_score: Optional[float]  # ADD THIS

In [31]:
def grounding_node(state: GenomicArbiterState):
    print("📍 [NODE 1] Grounding Physical Coordinates...")
    result = ensembl_gene_lookup.invoke(state["gene_symbol"])
    return {"physical_map": result}

In [32]:
def discovery_node(state: GenomicArbiterState):
    print(f"🔎 [NODE 2] Discovering ClinVar IDs for {state['gene_symbol']}...")
    result = clinvar_lookup.invoke(state["gene_symbol"])
    return {"discovered_ids": result}

In [33]:
def clinical_audit_node(state: GenomicArbiterState):
    print(f"🩺 [NODE 3] Auditing Clinical Pathogenicity for {state['variant_id']}...")
    result = clinvar_details.invoke(state["variant_id"])
    return {"clinvar_report": result}

In [34]:
def conflict_resolution_node(state: GenomicArbiterState):
    print("🌐 [NODE 4] Discrepancy Detected. Querying 2026 Evidence...")
    query = f"{state['gene_symbol']} variant {state['variant_id']} functional impact 2026"
    
    # Parallel tool execution
    p_res = pubmed_tool.invoke(query)
    e_res = exa_retriever.invoke(query)
    t_res = medical_genomic_tool.invoke(query)
    
    findings = [f"PubMed: {p_res}", f"Exa 2026: {e_res}", f"Tavily Deep Search: {t_res}"]
    return {"research_findings": findings}

In [35]:
def router(state: GenomicArbiterState):
    # Detects the "Uncertain significance" vs "FUNC" conflict from your image
    if "No Pathogenic" in state["clinvar_report"] and state["evo2_prediction"] == "FUNC":
        return "conflict_resolution"
    return "synthesis"

In [36]:
# ...existing code...

def synthesis_node(state: GenomicArbiterState):
    """Node 5: Final synthesis and report generation"""
    print("📝 [NODE 5] Creating Final Clinical Report...")

    pruned_research = []
    for finding in state["research_findings"]:
        pruned_research.append(finding[:1000] + "..." if len(finding) > 1000 else finding)
    
    prompt = ChatPromptTemplate.from_template("""
        You are a Senior Genomic Bioinformatician. Synthesize a clinical report.
        
        ### CONTEXT:
        - Gene: {gene_symbol}
        - Physical Map: {physical_map}
        - ClinVar: {clinvar_report}
        - Evo2 Prediction: {evo2_prediction}
        - 2026 Evidence: {research_findings}
        
        ### TASK:
        Create a crisp Markdown report with these sections:
        1. Gene Overview
        2. Clinical Relevance (2026 findings)
        3. Conflicting Evidence (if any)
        4. Discrepancy Resolution: VUS vs FUNC
        5. Conclusion
        
        If there is a VUS vs FUNC discrepancy, explain how the 2026 research resolves it. 
        No emojis. Focus on evidence quality.
    """)
    
    synthesis_chain = prompt | llm

    final_output = synthesis_chain.invoke({
        "gene_symbol": state["gene_symbol"],
        "physical_map": state["physical_map"][:400],
        "clinvar_report": state["clinvar_report"][:800],
        "evo2_prediction": state["evo2_prediction"],
        "research_findings": "\n\n".join(pruned_research)
    })

    return {"final_audit_report": final_output.content}

In [37]:
# ...existing code...

def clinical_actionability_node(state: GenomicArbiterState):
    """Node 6: Generate clinical recommendations and confidence scoring"""
    print("🏥 [NODE 6] Generating Clinical Recommendations...")
    
    # Calculate confidence score
    confidence = calculate_confidence(state)
    
    # Generate actionable recommendations
    prompt = ChatPromptTemplate.from_template("""
        You are a Clinical Genomics Consultant. Based on the genomic analysis, provide ACTIONABLE clinical recommendations.
        
        ### INPUT DATA:
        - Gene: {gene_symbol}
        - ClinVar Status: {clinvar_report}
        - Evo2 Prediction: {evo2_prediction}
        - 2026 Research: {research_summary}
        - Confidence Score: {confidence}/10
        
        ### TASK:
        Generate a structured Clinical Recommendations section with:
        
        1. **Testing Recommendations** (Who should be tested? When?)
        2. **Monitoring Protocols** (What biomarkers? How often?)
        3. **Therapeutic Implications** (Any treatment modifications?)
        4. **Patient Counseling Points** (What should patients know?)
        5. **Limitations & Contraindications** (What NOT to do based on evidence)
        
        Use bullet points. Be specific and evidence-based. No generic advice.
    """)
    
    chain = prompt | llm
    
    # Summarize research findings for context
    research_summary = "\n".join([
        finding[:300] + "..." 
        for finding in state["research_findings"][:2]  # Top 2 findings only
    ])
    
    recommendations = chain.invoke({
        "gene_symbol": state["gene_symbol"],
        "clinvar_report": state["clinvar_report"][:400],
        "evo2_prediction": state["evo2_prediction"],
        "research_summary": research_summary,
        "confidence": confidence
    })
    
    return {
        "clinical_recommendations": recommendations.content,
        "confidence_score": confidence
    }


def calculate_confidence(state: GenomicArbiterState) -> float:
    """Calculate confidence score based on evidence quality"""
    score = 5.0  # Baseline
    
    # Factor 1: ClinVar pathogenicity (+3 points)
    if "🚨 HIGH PRIORITY" in state.get("clinvar_report", ""):
        score += 3.0
    elif "No Pathogenic" in state.get("clinvar_report", ""):
        score -= 1.0
    
    # Factor 2: Research depth (+2 points)
    if len(state.get("research_findings", [])) >= 3:
        score += 2.0
    
    # Factor 3: Evo2 functional prediction (+1 point)
    if state.get("evo2_prediction") == "FUNC":
        score += 1.0
    
    # Factor 4: Research quality indicators
    research_text = " ".join(state.get("research_findings", []))
    if "replication" in research_text.lower():
        score += 0.5
    if "mechanism" in research_text.lower():
        score += 0.5
    if any(term in research_text.lower() for term in ["P = ", "OR = ", "CI"]):
        score += 1.0
    
    return min(10.0, max(1.0, score))  # Clamp between 1-10

In [42]:
# ...existing code...
builder = StateGraph(GenomicArbiterState)

# Add nodes
builder.add_node("grounding_node", grounding_node)
builder.add_node("discovery_node", discovery_node)
builder.add_node("clinical_audit_node", clinical_audit_node)
builder.add_node("conflict_resolution", conflict_resolution_node)
builder.add_node("synthesis", synthesis_node)
builder.add_node("clinical_actionability", clinical_actionability_node)  # NEW

# Add edges
builder.add_edge(START, "grounding_node")
builder.add_edge("grounding_node", "discovery_node")
builder.add_edge("discovery_node", "clinical_audit_node")
builder.add_conditional_edges("clinical_audit_node", router, {
    "conflict_resolution": "conflict_resolution",
    "synthesis": "synthesis"
})
builder.add_edge("conflict_resolution", "synthesis")
builder.add_edge("synthesis", "clinical_actionability")  # NEW
builder.add_edge("clinical_actionability", END)  # NEW

graph = builder.compile()

In [50]:
initial_input = {
    "gene_symbol": "BRAP",
    "variant_id": "4646508",
    "evo2_prediction": "FUNC",
    "research_findings": []
}

print("🚀 Starting Genomic Arbiter Pipeline...\n")

# Use invoke() to get complete final state
final_state = graph.invoke(initial_input)

# Print structured output
print("\n" + "="*80)
print("FINAL AUDIT REPORT")
print("="*80)
print(final_state.get("final_audit_report", ""))

print("\n" + "="*80)
print("CLINICAL RECOMMENDATIONS")
print(f"Confidence Score: {final_state.get('confidence_score', 'N/A')}/10")
print("="*80)
print(final_state.get("clinical_recommendations", ""))

print("\n" + "="*80)
print("📊 EXECUTION SUMMARY")
print("="*80)
print(f"Gene: {final_state['gene_symbol']}")
print(f"Physical Location: {final_state['physical_map'].split(chr(10))[2]}")  # Extract chromosome line
print(f"ClinVar IDs Found: {final_state['discovered_ids'].split(':')[1].strip()}")
print(f"Confidence: {final_state.get('confidence_score', 'N/A')}/10")

🚀 Starting Genomic Arbiter Pipeline...

📍 [NODE 1] Grounding Physical Coordinates...
🔎 [NODE 2] Discovering ClinVar IDs for BRAP...
🩺 [NODE 3] Auditing Clinical Pathogenicity for 4646508...
🌐 [NODE 4] Discrepancy Detected. Querying 2026 Evidence...
📝 [NODE 5] Creating Final Clinical Report...
🏥 [NODE 6] Generating Clinical Recommendations...

FINAL AUDIT REPORT
<think>
Okay, let's tackle this clinical report for the BRAP gene. First, I need to start with the Gene Overview. The user provided the Ensembl ID, location on chromosome 12, and the strand. I should present that clearly. Also, mention that BRAP interacts with BRCA1, which is important for DNA repair. That's a key point.

Next, Clinical Relevance based on the 2026 findings. The Exa 2026 document mentions SNPs in BRAP linked to myocardial infarction in Asians. The Tavily search also points to associations with blood pressure in Japanese populations. I need to highlight these two studies. Also, note that the 2026 evidence is from 

In [86]:
from fpdf import FPDF

def export_report(final_state):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 10, final_state["final_audit_report"])
    pdf.output("BRAP_Clinical_Report.pdf")

In [89]:
from fpdf import FPDF
from datetime import datetime

def clean_text_for_pdf(text: str) -> str:
    """Remove or replace Unicode characters that FPDF can't handle"""
    # Replace common Unicode characters with ASCII equivalents
    replacements = {
        '\u2013': '-',  # en dash
        '\u2014': '--',  # em dash
        '\u2018': "'",  # left single quote
        '\u2019': "'",  # right single quote
        '\u201c': '"',  # left double quote
        '\u201d': '"',  # right double quote
        '\u2022': '*',  # bullet point
        '\u2026': '...',  # ellipsis
        '\u00a0': ' ',  # non-breaking space
    }
    
    for unicode_char, ascii_char in replacements.items():
        text = text.replace(unicode_char, ascii_char)
    
    # Remove any remaining non-latin1 characters
    return text.encode('latin-1', errors='ignore').decode('latin-1')

def export_report(final_state):
    """Export genomic analysis report to PDF"""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    
    # Add title
    pdf.set_font("Arial", 'B', 16)
    pdf.cell(0, 10, f"Genomic Analysis Report: {final_state['gene_symbol']}", ln=True, align='C')
    pdf.ln(5)
    
    # Add metadata
    pdf.set_font("Arial", size=10)
    pdf.cell(0, 6, f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=True)
    pdf.cell(0, 6, f"Confidence Score: {final_state.get('confidence_score', 'N/A')}/10", ln=True)
    pdf.ln(5)
    
    # Add audit report
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 8, "Clinical Audit Report", ln=True)
    pdf.set_font("Arial", size=12)
    
    # Clean text before adding to PDF
    audit_report = clean_text_for_pdf(final_state.get("final_audit_report", "No report available"))
    pdf.multi_cell(0, 5, audit_report)
    pdf.ln(5)
    
    # Add recommendations
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 8, "Clinical Recommendations", ln=True)
    pdf.set_font("Arial", size=12)
    
    # Clean text before adding to PDF
    recommendations = clean_text_for_pdf(final_state.get("clinical_recommendations", "No recommendations available"))
    pdf.multi_cell(0, 5, recommendations)
    
    # Save PDF
    filename = f"{final_state['gene_symbol']}_Clinical_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    pdf.output(filename)
    print(f"\n✅ PDF Report saved: {filename}")
    return filename

# Execute pipeline and export
initial_input = {
    "gene_symbol": "BRAP",
    "variant_id": "4646508",
    "evo2_prediction": "FUNC",
    "research_findings": []
}

print("🚀 Starting Genomic Arbiter Pipeline...\n")

# Run pipeline
final_state = graph.invoke(initial_input)

# Print to console
print("\n" + "="*80)
print("FINAL AUDIT REPORT")
print("="*80)
print(final_state.get("final_audit_report", ""))

print("\n" + "="*80)
print("CLINICAL RECOMMENDATIONS")
print(f"Confidence Score: {final_state.get('confidence_score', 'N/A')}/10")
print("="*80)
print(final_state.get("clinical_recommendations", ""))

print("\n" + "="*80)
print("📊 EXECUTION SUMMARY")
print("="*80)
print(f"Gene: {final_state['gene_symbol']}")
print(f"Physical Location: {final_state['physical_map'].split(chr(10))[2]}")
print(f"ClinVar IDs Found: {final_state['discovered_ids'].split(':')[1].strip()}")
print(f"Confidence: {final_state.get('confidence_score', 'N/A')}/10")

# Export to PDF
export_report(final_state)

🚀 Starting Genomic Arbiter Pipeline...

📍 [NODE 1] Grounding Physical Coordinates...
🔎 [NODE 2] Discovering ClinVar IDs for BRAP...
🩺 [NODE 3] Auditing Clinical Pathogenicity for 4646508...
🌐 [NODE 4] Discrepancy Detected. Querying 2026 Evidence...
📝 [NODE 5] Creating Final Clinical Report...
🏥 [NODE 6] Generating Clinical Recommendations...

FINAL AUDIT REPORT
<think>
Okay, let's tackle this query. The user wants a clinical report on the BRAP gene. First, I need to parse all the provided information carefully. 

Starting with the Gene Overview. The gene is BRAP, located on chromosome 12, with the Ensembl ID given. The strand is reverse. I should mention the physical location and the Ensembl ID. Also, from the Tavily Deep Search, there's info about its role in protein ubiquitination and some associations with diseases like ischemic stroke and pulmonary hypertension. Need to include that.

Next, Clinical Relevance based on 2026 findings. The user mentioned Exa 2026 has a document about 

'BRAP_Clinical_Report_20260201_002722.pdf'

In [90]:
# ============================================================================
# 🧬 GENOMIC ARBITER - COMPREHENSIVE TEST SUITE
# ============================================================================

def run_genomic_test(gene_symbol: str, variant_id: str, evo2_prediction: str, test_name: str):
    """
    Run a complete genomic analysis pipeline test
    
    Args:
        gene_symbol: Gene name (e.g., "BRCA1")
        variant_id: ClinVar variant ID
        evo2_prediction: Functional prediction (FUNC/NONFUNC)
        test_name: Descriptive test name
    """
    print("\n" + "="*80)
    print(f"🧪 TEST CASE: {test_name}")
    print("="*80)
    
    initial_input = {
        "gene_symbol": gene_symbol,
        "variant_id": variant_id,
        "evo2_prediction": evo2_prediction,
        "research_findings": []
    }

    print(f"\n🚀 Starting Genomic Arbiter Pipeline for {gene_symbol}...\n")

    # Run pipeline
    final_state = graph.invoke(initial_input)

    # Print structured output
    print("\n" + "-"*80)
    print("📄 FINAL AUDIT REPORT")
    print("-"*80)
    print(final_state.get("final_audit_report", "")[:800] + "...\n")  # First 800 chars

    print("-"*80)
    print("💊 CLINICAL RECOMMENDATIONS")
    print(f"Confidence Score: {final_state.get('confidence_score', 'N/A')}/10")
    print("-"*80)
    print(final_state.get("clinical_recommendations", "")[:600] + "...\n")  # First 600 chars

    print("-"*80)
    print("📊 EXECUTION SUMMARY")
    print("-"*80)
    print(f"✓ Gene: {final_state['gene_symbol']}")
    print(f"✓ Physical Location: {final_state['physical_map'].split(chr(10))[2]}")
    print(f"✓ ClinVar IDs Found: {final_state['discovered_ids'].split(':')[1].strip()}")
    print(f"✓ Confidence: {final_state.get('confidence_score', 'N/A')}/10")
    print(f"✓ Conflict Resolution: {'✓ TRIGGERED' if 'No Pathogenic' in final_state.get('clinvar_report', '') else '✗ SKIPPED'}")

    # Export to PDF
    filename = export_report(final_state)
    print(f"✓ PDF Exported: {filename}")
    
    return final_state


# ============================================================================
# TEST CASE 1: BRAP (Cardiovascular Risk - Your Original Test)
# ============================================================================

print("\n" + "🔬"*40)
print("STARTING COMPREHENSIVE GENOMIC TESTING SUITE")
print("🔬"*40)

test1_result = run_genomic_test(
    gene_symbol="BRAP",
    variant_id="4646508",
    evo2_prediction="FUNC",
    test_name="BRAP - Cardiovascular Risk (VUS vs FUNC Conflict)"
)


# ============================================================================
# TEST CASE 2: BRCA1 (Breast Cancer - HIGH PRIORITY PATHOGENIC)
# ============================================================================

test2_result = run_genomic_test(
    gene_symbol="BRCA1",
    variant_id="17610",  # Known pathogenic variant
    evo2_prediction="FUNC",
    test_name="BRCA1 - Hereditary Breast Cancer (Pathogenic Variant)"
)


# ============================================================================
# TEST CASE 3: TP53 (Li-Fraumeni Syndrome - Another High-Risk Gene)
# ============================================================================

test3_result = run_genomic_test(
    gene_symbol="TP53",
    variant_id="376484",  # Known pathogenic variant in TP53
    evo2_prediction="FUNC",
    test_name="TP53 - Li-Fraumeni Syndrome (Multi-Cancer Risk)"
)


# ============================================================================
# COMPARATIVE ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("📊 COMPARATIVE ANALYSIS - ALL TESTS")
print("="*80)

tests = [
    ("BRAP", test1_result),
    ("BRCA1", test2_result),
    ("TP53", test3_result)
]

print(f"\n{'Gene':<10} {'Chromosome':<15} {'Confidence':<12} {'Priority':<20} {'Conflict?'}")
print("-"*80)

for gene, result in tests:
    chromosome = result['physical_map'].split('\n')[2].split('Chromosome ')[1].split(' ')[0]
    confidence = f"{result.get('confidence_score', 0):.1f}/10"
    priority = "🚨 HIGH" if "🚨 HIGH PRIORITY" in result.get('clinvar_report', '') else "⚠️ VUS/Benign"
    conflict = "✓ YES" if "No Pathogenic" in result.get('clinvar_report', '') and result.get('evo2_prediction') == "FUNC" else "✗ NO"
    
    print(f"{gene:<10} {chromosome:<15} {confidence:<12} {priority:<20} {conflict}")

print("\n" + "="*80)
print("✅ ALL TESTS COMPLETED")
print("="*80)
print(f"\nTotal PDFs Generated: 3")
print(f"- {test1_result['gene_symbol']}_Clinical_Report_*.pdf")
print(f"- {test2_result['gene_symbol']}_Clinical_Report_*.pdf")
print(f"- {test3_result['gene_symbol']}_Clinical_Report_*.pdf")


🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬
STARTING COMPREHENSIVE GENOMIC TESTING SUITE
🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬

🧪 TEST CASE: BRAP - Cardiovascular Risk (VUS vs FUNC Conflict)

🚀 Starting Genomic Arbiter Pipeline for BRAP...

📍 [NODE 1] Grounding Physical Coordinates...
🔎 [NODE 2] Discovering ClinVar IDs for BRAP...
🩺 [NODE 3] Auditing Clinical Pathogenicity for 4646508...
🌐 [NODE 4] Discrepancy Detected. Querying 2026 Evidence...
📝 [NODE 5] Creating Final Clinical Report...
🏥 [NODE 6] Generating Clinical Recommendations...

--------------------------------------------------------------------------------
📄 FINAL AUDIT REPORT
--------------------------------------------------------------------------------
<think>
Okay, let's tackle this query. The user wants a clinical report on the BRAP gene. First, I need to parse all the provided information carefully. 

Starting with the Gene Overview. The gene is BRAP, located on chromosome 12, with the Ensembl ID given. The strand 

In [91]:
# ============================================================================
# 🧬 PROSTATE CANCER GENOMIC ANALYSIS - TEST SUITE
# ============================================================================

def run_prostate_cancer_test(gene_symbol: str, variant_id: str, evo2_prediction: str, test_name: str):
    """
    Run prostate cancer genomic analysis pipeline
    
    Args:
        gene_symbol: Prostate cancer gene (e.g., "BRCA2", "ATM", "HOXB13")
        variant_id: ClinVar variant ID
        evo2_prediction: Functional prediction (FUNC/NONFUNC)
        test_name: Descriptive test name
    """
    print("\n" + "="*80)
    print(f"🧪 PROSTATE CANCER TEST: {test_name}")
    print("="*80)
    
    initial_input = {
        "gene_symbol": gene_symbol,
        "variant_id": variant_id,
        "evo2_prediction": evo2_prediction,
        "research_findings": []
    }

    print(f"\n🚀 Starting Prostate Cancer Genomic Analysis for {gene_symbol}...\n")

    # Run pipeline
    final_state = graph.invoke(initial_input)

    # Print structured output
    print("\n" + "-"*80)
    print("📄 PROSTATE CANCER AUDIT REPORT")
    print("-"*80)
    print(final_state.get("final_audit_report", "")[:900] + "...\n")

    print("-"*80)
    print("💊 PROSTATE CANCER TREATMENT RECOMMENDATIONS")
    print(f"Confidence Score: {final_state.get('confidence_score', 'N/A')}/10")
    print("-"*80)
    print(final_state.get("clinical_recommendations", "")[:700] + "...\n")

    print("-"*80)
    print("📊 PROSTATE CANCER ANALYSIS SUMMARY")
    print("-"*80)
    print(f"✓ Gene: {final_state['gene_symbol']}")
    print(f"✓ Physical Location: {final_state['physical_map'].split(chr(10))[2]}")
    print(f"✓ ClinVar IDs Found: {final_state['discovered_ids'].split(':')[1].strip()}")
    print(f"✓ Confidence: {final_state.get('confidence_score', 'N/A')}/10")
    print(f"✓ Conflict Resolution: {'✓ TRIGGERED' if 'No Pathogenic' in final_state.get('clinvar_report', '') else '✗ SKIPPED'}")
    
    # Check for PARP inhibitor eligibility
    if "BRCA" in gene_symbol or "ATM" in gene_symbol:
        print(f"✓ PARP Inhibitor Eligible: ✅ YES (Olaparib/Rucaparib indicated)")
    else:
        print(f"✓ PARP Inhibitor Eligible: ⚠️ Evaluate case-by-case")

    # Export to PDF
    filename = export_report(final_state)
    print(f"✓ PDF Exported: {filename}")
    
    return final_state


# ============================================================================
# TEST CASE 1: BRCA2 (Aggressive Prostate Cancer - FDA Approved PARP Target)
# ============================================================================

print("\n" + "🔬"*40)
print("PROSTATE CANCER GENOMIC TESTING SUITE - 2026")
print("🔬"*40)

print("\n" + "🎯"*40)
print("FOCUS: Hereditary Prostate Cancer & Precision Oncology")
print("🎯"*40)

test1_result = run_prostate_cancer_test(
    gene_symbol="BRCA2",
    variant_id="128144",  # Known pathogenic BRCA2 variant
    evo2_prediction="FUNC",
    test_name="BRCA2 - Metastatic Castration-Resistant Prostate Cancer (mCRPC)"
)


# ============================================================================
# TEST CASE 2: ATM (DNA Damage Response - PARP Inhibitor Target)
# ============================================================================

test2_result = run_prostate_cancer_test(
    gene_symbol="ATM",
    variant_id="475899",  # Known pathogenic ATM variant
    evo2_prediction="FUNC",
    test_name="ATM - Homologous Recombination Deficiency (HRD) Prostate Cancer"
)


# ============================================================================
# TEST CASE 3: HOXB13 (Hereditary Prostate Cancer Predisposition)
# ============================================================================

test3_result = run_prostate_cancer_test(
    gene_symbol="HOXB13",
    variant_id="41269",  # G84E founder mutation (highly penetrant)
    evo2_prediction="FUNC",
    test_name="HOXB13 - Early-Onset Hereditary Prostate Cancer (G84E Mutation)"
)


# ============================================================================
# TEST CASE 4: PTEN (Aggressive Disease & Poor Prognosis Marker)
# ============================================================================

test4_result = run_prostate_cancer_test(
    gene_symbol="PTEN",
    variant_id="433067",  # Known pathogenic PTEN variant
    evo2_prediction="FUNC",
    test_name="PTEN - Aggressive Gleason Score & Metastatic Risk"
)


# ============================================================================
# PROSTATE CANCER COMPARATIVE ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("📊 PROSTATE CANCER GENOMIC ANALYSIS - COMPARATIVE RESULTS")
print("="*80)

prostate_tests = [
    ("BRCA2", test1_result, "🎯 PARP Inhibitor"),
    ("ATM", test2_result, "🎯 PARP Inhibitor"),
    ("HOXB13", test3_result, "🔍 Surveillance"),
    ("PTEN", test4_result, "⚠️ Aggressive")
]

print(f"\n{'Gene':<10} {'Chromosome':<15} {'Confidence':<12} {'Therapy':<20} {'Priority':<15}")
print("-"*80)

for gene, result, therapy in prostate_tests:
    chromosome = result['physical_map'].split('\n')[2].split('Chromosome ')[1].split(' ')[0]
    confidence = f"{result.get('confidence_score', 0):.1f}/10"
    priority = "🚨 HIGH" if "🚨 HIGH PRIORITY" in result.get('clinvar_report', '') else "⚠️ VUS/Monitor"
    
    print(f"{gene:<10} {chromosome:<15} {confidence:<12} {therapy:<20} {priority:<15}")

print("\n" + "="*80)
print("🧬 PROSTATE CANCER THERAPEUTIC INSIGHTS")
print("="*80)

# Calculate PARP inhibitor eligibility
parp_eligible = []
for gene, result, therapy in prostate_tests:
    if "PARP" in therapy:
        parp_eligible.append(gene)

print(f"\n✅ PARP Inhibitor Eligible Genes: {', '.join(parp_eligible)}")
print(f"   → FDA Approved: Olaparib (Lynparza), Rucaparib (Rubraca)")
print(f"   → Indication: mCRPC with BRCA1/2 or ATM mutations")
print(f"   → Response Rate: ~30-50% (TRITON2/PROfound trials)")

print(f"\n⚠️  High-Risk Surveillance Genes: HOXB13, PTEN")
print(f"   → Annual PSA screening from age 40")
print(f"   → Consider prostate MRI at elevated PSA")
print(f"   → Genetic counseling for family members")

print("\n" + "="*80)
print("✅ PROSTATE CANCER GENOMIC ANALYSIS COMPLETED")
print("="*80)
print(f"\nTotal PDFs Generated: 4")
print(f"- {test1_result['gene_symbol']}_Clinical_Report_*.pdf (BRCA2 - PARP Target)")
print(f"- {test2_result['gene_symbol']}_Clinical_Report_*.pdf (ATM - PARP Target)")
print(f"- {test3_result['gene_symbol']}_Clinical_Report_*.pdf (HOXB13 - Hereditary)")
print(f"- {test4_result['gene_symbol']}_Clinical_Report_*.pdf (PTEN - Aggressive)")

print("\n" + "="*80)
print("🎯 CLINICAL DECISION SUPPORT")
print("="*80)
print("""
KEY ACTIONABLE INSIGHTS:

1. **BRCA2/ATM Mutations** → Consider PARP inhibitors (FDA approved for mCRPC)
   - Olaparib: 300mg BID (PROfound trial: 7.4 vs 3.6 months rPFS)
   - Rucaparib: 600mg BID (TRITON2 trial: 44% ORR)

2. **HOXB13 G84E** → Enhanced screening protocol
   - Start PSA at age 40 (vs 50 standard)
   - Annual screening (vs biennial)
   - Consider prostate MRI if PSA >2.5 ng/mL

3. **PTEN Loss** → Aggressive disease marker
   - Higher Gleason scores (≥8)
   - Increased metastatic risk
   - Consider platinum-based chemotherapy

4. **Germline Testing** → Cascade testing indicated
   - 50% transmission risk to offspring
   - Inform family members (especially male relatives)
   - Consider genetic counseling referral
""")

print("\n" + "🏥"*40)
print("END OF PROSTATE CANCER GENOMIC ANALYSIS")
print("🏥"*40)


🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬
PROSTATE CANCER GENOMIC TESTING SUITE - 2026
🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬🔬

🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯
FOCUS: Hereditary Prostate Cancer & Precision Oncology
🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯

🧪 PROSTATE CANCER TEST: BRCA2 - Metastatic Castration-Resistant Prostate Cancer (mCRPC)

🚀 Starting Prostate Cancer Genomic Analysis for BRCA2...

📍 [NODE 1] Grounding Physical Coordinates...
🔎 [NODE 2] Discovering ClinVar IDs for BRCA2...
🩺 [NODE 3] Auditing Clinical Pathogenicity for 128144...
📝 [NODE 5] Creating Final Clinical Report...
🏥 [NODE 6] Generating Clinical Recommendations...

--------------------------------------------------------------------------------
📄 PROSTATE CANCER AUDIT REPORT
--------------------------------------------------------------------------------
<think>
Okay, let's tackle this query. The user wants a clinical report on a BRCA2 gene variant, but wait, the ClinVar entry mentions PALB2. Hmm, n